# Notebook for prototyping individual blocks

In [1]:
# imports 
import math 
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F  

In [ ]:
# data class for architecture configurations 
@dataclass 
class Config:
    vocab_size: int        # size of token vocab (tokenizer.get_vocab_size())
    block_size: int = 256  # max context length for model to view
    n_layer: int = 6       # number of stacked transformer blocks
    n_head: int = 6        # attention heads per block (n_embd must divide)
    n_embd: int = 384      # width of the residual stream
    dropout: float = 0.1   # dropout proportion 
    

In [12]:
# data class for testing
small_config = lambda: Config(vocab_size=50, block_size=16, n_layer=2,
                              n_head=4, n_embd=32, dropout=0.0) 

## Positional Encoding

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, n_embd: int, block_size: int, dropout: float=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(block_size, n_embd) # (block_size, C)
        position = torch.arange(0, block_size).unsqueeze(1).float() # (block_size, 1)
        # frequencies: geometric spacing from 1 down to ~1/10000
        div_term = torch.exp(torch.arange(0, n_embd, 2).float()
                             * (-math.log(10000.0) / n_embd))
        pe[:,0::2] = torch.sin(position * div_term) # even -> sine
        pe[:,1::2] = torch.cos(position * div_term)  # odd  -> cosine
        self.register_buffer('pe', pe) # (block_size, C)

    def forward(self, x):
        # x = (B, T, C). add encoding for the first T positions and broadcast over
        # the batch dims.
        x = x + self.pe[: x.size(1)]
        return self.dropout(x)

In [16]:
# testing 
def test_positional_encoding():
    cfg = small_config()
    pe = PositionalEncoding(cfg.n_embd, cfg.block_size)
    x = torch.zeros(1, cfg.block_size, cfg.n_embd)
    y = pe(x)
    assert y.shape == x.shape       # check shape is unchanged
    assert not torch.allclose(y, x) # check something was actually added
    print('Positional Encoding OK')

test_positional_encoding()

Positional Encoding OK


## Multi-Head Attention 

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        assert config.n_embd % config.n_head == 0, \
            'n_embd must be divisible by n_head'
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        # query/key/value projections
        self.query = nn.Linear(config.n_embd, config.n_embd)
        self.key   = nn.Linear(config.n_embd, config.n_embd)
        self.value = nn.Linear(config.n_embd, config.n_embd)

        self.proj  = nn.Linear(config.n_embd, config.n_embd) # combines heads
        self.dropout = nn.Dropout(config.dropout)

        # lower-triangluar mask
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('mask', mask.view(1,1,config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape

        # project, then reshape into heads: (B,T,C) -> (B, n_head, T, head_dim)
        q = self.query(x).view(B, T, self.n_head, self.head_dim).transpose(1,2)
        k = self.key(x).view(B, T, self.n_head, self.head_dim).transpose(1,2)
        v = self.value(x).view(B, T, self.n_head, self.head_dim).transpose(1,2)

        # attention scores scaled by 1/sqrt(head_dim)
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        # block out the future then normalize into probabilities
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v # (B, n_head, T, head_dim)
        y = y.transpose(1,2).contiguous().view(B,T,C) # recombine heads
        return self.proj(y)

In [20]:
# testing
def test_multihead():
    cfg = small_config()
    attn = MultiHeadAttention(cfg).eval()
    x = torch.randn(2, cfg.block_size, cfg.n_embd)
    y = attn(x)
    assert y.shape == x.shape # check shape is unchanged

    # causal check: last token should not affect earlier outputs
    x2 = x.clone()
    x2[:, -1, :] += 1.0
    assert torch.allclose(y[:, :-1], attn(x2)[:, :-1], atol=1e-5)
    print('Multi-Head Attention OK' )

test_multihead()


Multi-Head Attention OK


## Position-wise Feed Forward

In [26]:
class PosWiseFeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4*config.n_embd),
            nn.GELU(),
            nn.Linear(4*config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )
    def forward(self, x):
        return self.net(x)

In [27]:
# testing
def test_feed_forward():
    cfg = small_config()
    ff = PosWiseFeedForward(cfg).eval()
    x = torch.randn(2, cfg.block_size, cfg.n_embd)
    assert ff(x).shape == x.shape
    print("PositionWiseFeedForward OK")

test_feed_forward()

PositionWiseFeedForward OK


## Block, one transformer layer

In [ ]:
class Block(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = PosWiseFeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))
    

In [28]:
# testing
def test_block():
    cfg = small_config()
    blk = Block(cfg).eval()
    x = torch.randn(2, cfg.block_size, cfg.n_embd)
    assert blk(x).shape == x.shape # shape checks
    print('Block OK')

test_block()

Block OK


## Transformer class

In [36]:
class Transformer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_encoding = PositionalEncoding(config.n_embd, config.block_size, config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

        self.apply(self._init_weights)
        print(f"model parameters: {sum(p.numel() for p in self.parameters()) / 1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, \
        f"sequence length {T} excceds block_size {self.config.block_size}"

        x = self.token_embedding(idx) # (B, T, C)
        x = self.pos_encoding(x) # add position information
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x) # (B, T, V)

        loss = None
        if targets is not None:
            # compare preds at every position to the true next token
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Sample tokens one at a time, feeding each back in. idx: (B, T0)."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]   # keep the last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature       # focus on the final position
            if top_k is not None:                          # optionally keep only top-k
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx





In [37]:
# testing
def test_transformer():
    cfg = small_config()
    model = Transformer(cfg).eval()
    idx = torch.randint(0, cfg.vocab_size, (2, cfg.block_size))
    logits, loss = model(idx, targets=idx)
    assert logits.shape == (2, cfg.block_size, cfg.vocab_size)
    assert loss.item() > 0                 # cross-entropy is positive
    # generation adds exactly max_new_tokens to the context
    out = model.generate(idx[:, :4], max_new_tokens=5)
    assert out.shape == (2, 9)
    print("Transformer OK")

test_transformer()


model parameters: 0.03M
Transformer OK
